In [2]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.linear_model import LinearRegression, LogisticRegressionCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

In [ ]:
track_history  = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/now_playing.csv")
track_analysis = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_analysis.csv")
track_metadata = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_metadata.csv")
weather        = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/weather.csv")

# convert all request times to datetime formats
for df in [track_history, track_analysis, track_metadata, weather]:
    df["request_time"] = pd.to_datetime(df["request_time"])

# converts a string in the format mm:ss to milliseconds
def mm_ss_to_ms(mm_ss_str):
    minutes, seconds = map(int, mm_ss_str.split(":"))
    return ((minutes * 60) + seconds) * 1000

# Clean (types, missing values, odd values)
# Project: keep the columns (and rows) you need
# Combine tables / change what a row is (if needed)

# remove analysis/metadata observations whose id is null
track_analysis = track_analysis[~track_analysis["id"].isna()]
track_metadata = track_metadata[~track_metadata["id"].isna()]
# remove all metadata rows whose genres are null
track_metadata = track_metadata.dropna(subset=["genres"])
# if any values are missing in the track metadata (which is acceptable), set them to be an empty string
track_metadata.fillna("", inplace=True)
# change track analysis duration (mm:ss) to milliseconds
track_analysis["duration"] = track_analysis["duration"].apply(lambda d: mm_ss_to_ms(d))
# remove dB from track analysis loudness and convert to an int (e.g., "-2 dB" becomes -2)
track_analysis["loudness"] = track_analysis["loudness"].apply(lambda db: db.replace(" dB", "")).astype(np.int16)
# split camelot into three columns: X/y position via cos(camelot num * 2pi/12) and is_major
track_analysis["is_major"] = (track_analysis["camelot"].apply(lambda camelot: "B" in camelot)).astype(np.int16)
# turn camelot into coordinates (better for modeling; see explanation below)
track_analysis["camelot_x"] = np.cos(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)
track_analysis["camelot_y"] = np.sin(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)

# keep only relevant features
track_analysis = track_analysis[["request_id", "request_time", "mode", "camelot", "camelot_x", "camelot_y", "is_major", "duration", "tempo", "popularity", "energy", "happiness", "acousticness", "danceability", "instrumentalness", "speechiness", "liveness", "loudness"]]
track_history  = track_history[["request_id", "request_time", "title", "artist", "album", "duration", "program_start_ts", "plays", "is_ad"]]
track_metadata = track_metadata.drop(columns=["id", "artist_sort_name", "release_statuses", "release_languages", "release_scripts", "release_qualities"])

# separate ads and actual songs played into separate dataframes
track_ad_history = track_history[track_history["is_ad"]]
track_history    = track_history[~track_history["is_ad"]]

# one hot encode individual genres
# TODO: mostly sparse...
genres_df = pd.concat([track_metadata[["request_id", "request_time"]], track_metadata["genres"].str.get_dummies(sep=" | ")], axis=1)

# combine tables
data = pd.merge(track_analysis, track_metadata, on=["request_id", "request_time"], how="inner")
data = pd.merge(data, track_history, on=["request_id", "request_time"], how="inner")
data = pd.merge(data.drop("genres", axis=1), genres_df, on=["request_id", "request_time"], how="inner")
data

,request_id,request_time,mode,camelot,camelot_x,camelot_y,is_major,duration_x,tempo,popularity,...,techno,trance,trap,trip,tropicália,twee,urban,vaporwave,wave,|
0,01a15c64-a327-706e-aeff-3157601dc51f,2026-10-21 02:57:41.671393+00:00,minor,10A,5.000000e-01,-8.660254e-01,0,188000,131,56,...,0,0,0,0,0,0,0,0,0,0
1,01a15c73-d8a6-747f-aa75-cb4fb88031f9,2026-10-21 03:14:18.406577+00:00,major,3B,6.123234e-17,1.000000e+00,1,234000,84,46,...,0,0,0,0,0,0,0,0,0,0
2,01a15cec-dd59-73fc-b30c-31f9cf2a0d89,2026-10-21 05:26:29.465178+00:00,major,3B,6.123234e-17,1.000000e+00,1,194000,108,64,...,0,0,0,0,0,0,0,0,0,0
3,01a15d13-2596-77d0-9f2b-c07761a8e47f,2026-10-21 06:08:18.326578+00:00,major,3B,6.123234e-17,1.000000e+00,1,201000,89,64,...,0,0,0,0,0,0,0,0,0,0
4,01a15d15-9ced-72e0-9e2f-186d6be8fd86,2026-10-21 06:10:59.949102+00:00,major,7B,-8.660254e-01,-5.000000e-01,1,182000,170,50,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,01a19df1-f4da-714b-88fb-67350f50ba2a,2026-11-02 20:27:22.202981+00:00,major,10B,5.000000e-01,-8.660254e-01,1,212000,131,83,...,0,0,0,0,0,0,0,0,0,0
668,01a19e05-53dd-7706-b213-b5d023ebb5be,2026-11-02 20:48:31.709558+00:00,major,11B,8.660254e-01,-5.000000e-01,1,222000,178,55,...,0,0,0,0,0,0,0,0,0,0
669,01a19e0f-ef25-725f-8c3a-3194e6b01118,2026-11-02 21:00:06.821552+00:00,major,12B,1.000000e+00,-2.449294e-16,1,187000,126,47,...,0,0,0,0,0,0,0,0,0,0
670,01a19e1b-7638-7278-aa99-8ed9d2d80913,2026-11-02 21:12:42.296976+00:00,major,7B,-8.660254e-01,-5.000000e-01,1,211000,93,68,...,0,0,0,0,0,0,0,0,0,0


The camelot wheel is a circle, so standard linear numbers can confuse ML models. Specifically, a model might think 12 and 1 are far apart, even though they are neighbors. The solution used here is projecting the 12 positions onto a 2D coordinate system using cos/sin.

For example, 12 and 1 on the camelot wheel are in key (and should therefore have a small "distance"). 12 lies at point (1, 0); 1 at (.866, 0.5). Using euclidian distance, the distance between them is 0.51; the distance between 12 and 6 (opposite sides of the wheel) is 2.0, which we can use to describe that that 12 and 1 are very closely related, while 12 and 6 are mathematically distant (opposites on the camelot wheel).